**DATA VALIDATION QUALITY CHECKS**
# Data Validation & Quality Checks

This notebook performs lightweight validation checks on Gold tables:
- Row counts
- Null rates on key fields
- Duplicate key detection

Purpose: sanity checks before reporting and refresh.


****CELL ROW COUNT**


In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
tables = [
    "gold_dim_customer",
    "gold_fact_orders",
    "gold_fact_payments",
    "gold_fact_support",
    "gold_fact_web",
    "gold_customer360_summary"
]

row_counts = []

for t in tables:
    cnt = spark.table(t).count()
    row_counts.append((t, cnt))

row_counts_df = spark.createDataFrame(row_counts, ["table_name", "row_count"])
display(row_counts_df)


StatementMeta(, 9a7bd1ee-25c1-479e-b3ce-e8820f196970, 3, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7b2ff11f-6e3d-4279-93d8-6c998392758a)

**Code Cell Null checks (critical fields)**

In [2]:
from pyspark.sql.functions import col, sum

null_checks = (
    spark.table("gold_customer360_summary")
    .select(
        sum(col("customer_id").isNull().cast("int")).alias("null_customer_id"),
        sum(col("email").isNull().cast("int")).alias("null_email"),
        sum(col("order_count").isNull().cast("int")).alias("null_order_count"),
        sum(col("avg_order_value").isNull().cast("int")).alias("null_avg_order_value"),
        sum(col("is_active_customer").isNull().cast("int")).alias("null_is_active_customer")
    )
)

display(null_checks)


StatementMeta(, 9a7bd1ee-25c1-479e-b3ce-e8820f196970, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, c3f3936b-a407-4d2f-8069-de3a36e4b06f)

**Code Cell Duplicate Key Check: TABLE SHOULD BE EMPTY**

In [3]:
dup_check = (
    spark.table("gold_customer360_summary")
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
)

display(dup_check)


StatementMeta(, 9a7bd1ee-25c1-479e-b3ce-e8820f196970, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, b6035868-d2d9-410d-8be9-a0edfd34562b)